## Imports and Setup

In [ ]:
%pip install -q rapidfuzz

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
import re

from rapidfuzz import process, fuzz

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

from sklearn.metrics.pairwise import PAIRWISE_KERNEL_FUNCTIONS

from xgboost import XGBClassifier

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataset Loading

**Outcome:** at least **3 distinct households** each watch **at least one-third** of the title runtime **within 6 months** after acquisition. Only titles with a complete eligible six-month observation window are labelled for supervised learning. Titles with incomplete follow-up remain unlabelled.

In [ ]:
# Load data

viewing = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vR47OdFiWY13aDvTZN4QIbiMODrZlyDQ29q6Jgn2BrpofN1ZjmbJ4X95U8RrmssMd5PVD8AVvpjKo95/pub?gid=737932283&single=true&output=csv")
catalogue = pd.read_csv( "https://docs.google.com/spreadsheets/d/e/2PACX-1vTp01l3Uiqy4c_B9AT1q-cWpwDa-vphrvDeT0etTu6Xe4XHBEFoJjJzj9IUZL-V2K7BH9_LyyU5rJHJ/pub?gid=1759543568&single=true&output=csv")
tmdb = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vTREHT79TLNL9A_G8mbWvessC2JcmOLgfiPRPty_i1yKuo-Rf5191D2XUREM4Q7x5R7NncdNzGUdxUF/pub?gid=386996102&single=true&output=csv")

# Check for missing values
print("\nMissing values:")
display(viewing.isna().sum().to_frame("Viewing"))
display(catalogue.isna().sum().to_frame("Catalogue"))
display(tmdb.isna().sum().to_frame("TMDB"))


Missing values:


,Viewing
household_id,0
session_id,0
event_timestamp,0
title_id,8853
event_type,0
watch_duration,9157
title_runtime,0


,Catalogue
title_id,0
title_name,0
acquisition_date,0
content_type,0
licensing_cost_usd,0
license_expiry_date,0
release_year,0
new_title,0


,TMDB
tmdb_id,0
stadio_title_id,0
genres,0
lead_actors,0
user_score,0
original_language,0
network,0


## Dataset Preprocessing

In [ ]:
viewing.info()
catalogue.info()
tmdb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45305 entries, 0 to 45304
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   household_id     45305 non-null  object 
 1   session_id       45305 non-null  object 
 2   event_timestamp  45305 non-null  int64  
 3   title_id         36452 non-null  object 
 4   event_type       45305 non-null  object 
 5   watch_duration   36148 non-null  float64
 6   title_runtime    45305 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 2.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1140 entries, 0 to 1139
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   title_id             1140 non-null   object 
 1   title_name           1140 non-null   object 
 2   acquisition_date     1140 non-null   object 
 3   content_type         1140 non-null   object 
 4   licen

In [ ]:

# make primary key the same by stripping any additional characters and lowering the key

viewing['title_id'] = viewing['title_id'].str.strip().str.lower()
catalogue['title_id'] = catalogue['title_id'].str.strip().str.lower()
tmdb["stadio_title_id"] = tmdb["stadio_title_id"].str.strip().str.lower()

# transforming data types
viewing["event_type"] = viewing["event_type"].str.strip().str.lower()
viewing["event_date"] = pd.to_datetime(viewing["event_timestamp"],unit="s",utc=True).dt.tz_localize(None)
catalogue["acquisition_date"] = pd.to_datetime(catalogue["acquisition_date"],errors="coerce")
catalogue["license_expiry_date"] = pd.to_datetime(catalogue["license_expiry_date"],errors="coerce")

print("Datatype after conversion")
print("Viewing events:")
viewing.info()
print("####################")
print("Catalogue titles:")
catalogue.info()
print("####################")
print("Metadata records:")
tmdb.info()

Datatype after conversion
Viewing events:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45305 entries, 0 to 45304
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   household_id     45305 non-null  object        
 1   session_id       45305 non-null  object        
 2   event_timestamp  45305 non-null  int64         
 3   title_id         36452 non-null  object        
 4   event_type       45305 non-null  object        
 5   watch_duration   36148 non-null  float64       
 6   title_runtime    45305 non-null  int64         
 7   event_date       45305 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 2.8+ MB
####################
Catalogue titles:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1140 entries, 0 to 1139
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               -----------

In [ ]:
# removing duplicates

viewing = viewing.drop_duplicates()
catalogue = catalogue.drop_duplicates("title_id")
tmdb = tmdb.drop_duplicates("stadio_title_id")

print("Dataframe lengths after dedupe")
print("Viewing events:", len(viewing))
print("Catalogue titles:", len(catalogue))
print("Metadata records:", len(tmdb))

Dataframe lengths after dedupe
Viewing events: 45304
Catalogue titles: 1140
Metadata records: 1076


In [ ]:
# Six-month engagement observation window.

OBSERVATION_MONTHS = 6

log_start = pd.Timestamp("2023-09-01")
log_end = pd.Timestamp("2026-09-01")

catalogue["outcome_end"] = (
    catalogue["acquisition_date"]
    + pd.DateOffset(months=OBSERVATION_MONTHS)
)

cohort = catalogue[
    (catalogue["acquisition_date"] >= log_start)
    & (catalogue["outcome_end"] <= log_end)
    & (
        catalogue["license_expiry_date"]
        + pd.Timedelta(days=1)
        >= catalogue["outcome_end"]
    )
].copy()

print("Eligible titles:", len(cohort))

Eligible titles: 465


In [ ]:
# Keep only events containing valid viewing durations.

plays = viewing[viewing["event_type"].isin(["play", "pause"])& viewing["title_id"].notna()& viewing["watch_duration"].notna()& (viewing["watch_duration"] >= 0)& (viewing["title_runtime"] > 0)].copy()
plays = plays.merge(cohort[["title_id","acquisition_date","outcome_end"]],on="title_id",how="inner")
plays = plays[(plays["event_date"] >= plays["acquisition_date"])& (plays["event_date"] < plays["outcome_end"])]


# The duration is cumulative WITHIN a session.
# Keep the maximum instead of summing Play/Pause events.

sessions = (plays.groupby(["title_id", "household_id", "session_id"],as_index=False).agg(session_watch_seconds=("watch_duration", "max"),title_runtime=("title_runtime", "max")))


# Sum across the household's separate viewing sessions.

households = (sessions.groupby(["title_id", "household_id"],as_index=False).agg(total_watch_seconds=("session_watch_seconds", "sum"),title_runtime=("title_runtime", "max")))

households["qualifies"] = (households["total_watch_seconds"]>= households["title_runtime"] / 3)


# Count qualifying households for each title.

qualifiers = (households.groupby("title_id")["qualifies"].sum().rename("qualifying_households").reset_index())

cohort = cohort.merge(qualifiers,on="title_id",how="left")

cohort["qualifying_households"] = (cohort["qualifying_households"].fillna(0).astype(int))

cohort["target"] = (cohort["qualifying_households"] >= 3).astype(int)

print("Eligible titles:", len(cohort))
print("Successful titles:", cohort["target"].sum())
print("Positive rate:", round(cohort["target"].mean(), 4))

display(cohort["target"].value_counts().sort_index().rename("titles").to_frame())

Eligible titles: 465
Successful titles: 121
Positive rate: 0.2602


,titles
target,
0,344
1,121


In [ ]:
# Split by acquisition date, keeping titles from the same acquisition date together.

dates = sorted(cohort["acquisition_date"].unique())
cutoffs = [dates[int(len(dates) * proportion)]for proportion in (0.60, 0.75, 0.85)]
cohort["role"] = np.select([cohort["acquisition_date"] < cutoffs[0],cohort["acquisition_date"] < cutoffs[1],cohort["acquisition_date"] < cutoffs[2]],["train","tune","calibration"],default="test")
display(cohort.groupby("role")["target"].agg(titles="count",positives="sum",positive_rate="mean"))
display(cohort.groupby("role")["acquisition_date"].agg(["min", "max"]))

,titles,positives,positive_rate
role,,,
calibration,51,14,0.274510
test,68,18,0.264706
train,273,73,0.267399
tune,73,16,0.219178


,min,max
role,,
calibration,2025-07-27,2025-10-27
test,2025-10-30,2026-02-28
train,2023-09-05,2025-02-15
tune,2025-02-18,2025-07-18


In [ ]:
# Prepare the metadata table

tmdb = tmdb.rename(columns={"stadio_title_id": "title_id"})
titles = catalogue.merge(tmdb,on="title_id",how="left",validate="one_to_one")
titles = titles.merge(cohort[["title_id", "target", "role"]],on="title_id",how="left",validate="one_to_one")
titles["role"] = titles["role"].fillna("unlabelled")
titles["target"] = titles["target"].fillna(-1).astype(int)
titles = titles.sort_values("title_id").reset_index(drop=True)
titles["node_id"] = np.arange(len(titles))


# Metadata features available when evaluating a title.

titles["title_age_at_acquisition"] = (titles["acquisition_date"].dt.year- titles["release_year"]).clip(lower=0)
titles["release_year_missing"] = (titles["release_year"].isna().astype(int))
titles["genre_list"] = (titles["genres"].fillna("").map(lambda value: sorted({genre.strip().lower()for genre in value.split(",")if genre.strip()})))
titles["genre_count"] = titles["genre_list"].map(len)
titles["lead_actor_count"] = (titles["lead_actors"].fillna("").map(lambda value: len([actor for actor in value.split(",")if actor.strip()])))
titles["tmdb_available_flag"] = (titles["tmdb_id"].notna().astype(int))
titles["new_title_flag"] = (titles["new_title"].astype(int))
titles["release_decade"] = (titles["release_year"].fillna(titles.loc[titles["role"] == "train","release_year"].median())// 10 * 10).astype(int)

for column in ["content_type","original_language","network"]:
    titles[column] = (titles[column].fillna("unknown").astype(str).str.strip().str.lower().replace("", "unknown"))

In [ ]:
def parse_actors(value):
    if pd.isna(value):
        return []

    return sorted({
        re.sub(r"\s+", " ", actor.strip()).casefold()
        for actor in str(value).split(",")
        if actor.strip()
    })


titles["actor_list"] = titles["lead_actors"].map(
    parse_actors
)


# Select actors using training titles only.

actor_counts = Counter(actor
for actors in titles.loc[
        titles["role"] == "train",
        "actor_list"
    ]
    for actor in actors
)

top_actors = [
    actor
    for actor, count in actor_counts.most_common(20)
]


# Multi-hot encoding: a title can contain multiple actors.

actor_flags = pd.DataFrame(
    {
        "actor_" + re.sub(
            r"[^a-z0-9]+", "_", actor
        ).strip("_"): titles["actor_list"].map(
            lambda actors, name=actor: float(name in actors)
        )
        for actor in top_actors
    },
    index=titles.index
)

Actor identity features: 20


,actor_idris_elba,actor_john_kani,actor_presley_chweneyagae,actor_sandra_prinsloo,actor_michael_fassbender,actor_rami_malek,actor_tessa_thompson,actor_neil_sandilands,actor_oscar_isaac,actor_colin_firth,actor_viola_davis,actor_chiwetel_ejiofor,actor_daniel_kaluuya,actor_rebecca_ferguson,actor_denzel_washington,actor_maggie_smith,actor_thuso_mbedu,actor_bonnie_mbuli,actor_sello_maake,actor_ryan_gosling
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Historical Viewing Feature

As I was building the notebook I realised that the model needed more feature. This cell build historical view features based off of similar titles for each title before it was aquired.

In [ ]:
#  Reconstruct historical viewing sessions.
#
# Use the FULL viewing dataset, rather than 'plays' or'cohort', which were restricted for target construction.

history_events = viewing.loc[
    viewing["event_type"].isin(["play", "pause"])
    & viewing["title_id"].notna()
    & viewing["household_id"].notna()
    & viewing["session_id"].notna()
].copy()

for col in ["watch_duration", "title_runtime"]:
    history_events[col] = pd.to_numeric(
        history_events[col],
        errors="coerce"
    )

history_events = history_events.loc[
    history_events["watch_duration"].notna()
    & (history_events["watch_duration"] >= 0)
    & (history_events["title_runtime"] > 0)
]


# Attach historical title information.

history_events = history_events.merge(
    titles[
        [
            "title_id",
            "acquisition_date",
            "license_expiry_date",
            "genre_list",
            "content_type"
        ]
    ],
    on="title_id",
    how="inner",
    validate="many_to_one"
)


# Only retain viewing activity inside the licensing period.

history_events = history_events.loc[
    (
        history_events["event_date"]
        >= history_events["acquisition_date"]
    )
    & (
        history_events["event_date"]
        < (
            history_events["license_expiry_date"]
            + pd.Timedelta(days=1)
        )
    )
].copy()


# Watch duration is cumulative within a session. Retain the maximum rather than summing Play/Pause events.

historical_sessions = (
    history_events
    .groupby(
        ["title_id", "household_id", "session_id"],
        as_index=False
    )
    .agg(
        session_end=("event_date", "max"),
        watch_seconds=("watch_duration", "max"),
        runtime_seconds=("title_runtime", "max"),
        genre_list=("genre_list", "first"),
        content_type=("content_type", "first")
    )
)


# Calculate session-level engagement. Cap viewing fraction at 100% for abnormal durations.

historical_sessions["watch_fraction"] = (
    historical_sessions["watch_seconds"]
    / historical_sessions["runtime_seconds"]
).clip(0, 1)

historical_sessions["completed_session"] = (
    historical_sessions["watch_fraction"] >= (1/3)
).astype(float)



# Create efficient historical lookup tables.

def build_lookup(df, key_col):

    result = {}

    for key, block in df.groupby(key_col, sort=False):

        block = block.sort_values("session_end")

        result[key] = {
            "time": block["session_end"].to_numpy(
                dtype="datetime64[ns]"
            ),

            "completed_cumsum": np.r_[
                0.,
                block["completed_session"]
                .to_numpy()
                .cumsum()
            ],

            "watch_cumsum": np.r_[
                0.,
                block["watch_fraction"]
                .to_numpy()
                .cumsum()
            ]
        }

    return result


# Overall platform history.

all_history = build_lookup(
    historical_sessions.assign(history_key="all"),
    "history_key"
)


# Viewing history by genre.

genre_history = build_lookup(
    historical_sessions
    .explode("genre_list")
    .dropna(subset=["genre_list"])
    .rename(columns={"genre_list": "history_key"}),
    "history_key"
)


# Viewing history by content type.

type_history = build_lookup(
    historical_sessions.rename(
        columns={"content_type": "history_key"}
    ),
    "history_key"
)



# Point-in-time feature calculations.

def prior_stats(
    lookup,
    key,
    asof,
    start_days,
    end_days=0
):

    """
    Returns:
        1. Session count
        2. Completed session count
        3. Sum of session watch fractions

    Uses only the period:
        [asof - start_days, asof - end_days)

    The acquisition date itself is excluded.
    """

    entry = lookup.get(key)

    if entry is None or pd.isna(asof):
        return np.zeros(3, dtype=float)

    dates = entry["time"]

    lo = np.searchsorted(
        dates,
        np.datetime64(
            asof - pd.Timedelta(days=start_days)
        ),
        side="left"
    )

    hi = np.searchsorted(
        dates,
        np.datetime64(
            asof - pd.Timedelta(days=end_days)
        ),
        side="left"
    )

    return np.array([
        hi - lo,

        (
            entry["completed_cumsum"][hi]
            - entry["completed_cumsum"][lo]
        ),

        (
            entry["watch_cumsum"][hi]
            - entry["watch_cumsum"][lo]
        )
    ], dtype=float)


# Account for partial history near the beginning of the logs. For example, September 2023 does not have a full 365-day history available.

history_start = viewing["event_date"].min().floor("D")


def available_days(asof, start_days, end_days=0):

    start = max(
        asof - pd.Timedelta(days=start_days),
        history_start
    )

    end = asof - pd.Timedelta(days=end_days)

    return max(
        (end - start).total_seconds() / 86400,
        0.0
    )


# Generate historical features for every catalogue title.


rows = []

# Smooth sparse genre/content-type rates towards the historical platform average.
SMOOTHING = 20.0

for row in titles[
    [
        "title_id",
        "acquisition_date",
        "genre_list",
        "content_type"
    ]
].itertuples(index=False):

    t = row.acquisition_date

    # Overall viewing history.

    global90 = prior_stats(
        all_history, "all", t, 90
    )

    global365 = prior_stats(
        all_history, "all", t, 365
    )

    overall_completion = (
        (global365[1] + 1)
        / (global365[0] + 2)
    )

    overall_fraction = (
        global365[2] / global365[0]
        if global365[0] > 0
        else 0.5
    )


    # Genre viewing history.

    g90 = [
        prior_stats(genre_history, g, t, 90)
        for g in row.genre_list
    ]

    gprev = [
        prior_stats(genre_history, g, t, 180, 90)
        for g in row.genre_list
    ]

    g365 = [
        prior_stats(genre_history, g, t, 365)
        for g in row.genre_list
    ]

    recent = (
        np.mean(g90, axis=0)
        if g90 else np.zeros(3)
    )

    previous = (
        np.mean(gprev, axis=0)
        if gprev else np.zeros(3)
    )

    genre365 = (
        np.mean(g365, axis=0)
        if g365 else np.zeros(3)
    )


    # Same-content-type history.

    content365 = prior_stats(
        type_history,
        row.content_type,
        t,
        365
    )


    # Correct for incomplete historical windows.

    days90 = available_days(t, 90)

    days_prev = available_days(t, 180, 90)

    days365 = available_days(t, 365)


    recent_rate = (
        recent[0] / max(days90, 1)
    )

    previous_rate = (
        previous[0] / max(days_prev, 1)
    )


    # Compare recent and preceding 90-day viewing rates. Neutral trend where there is insufficient history.

    trend = (
        (recent_rate + 0.05)
        / (previous_rate + 0.05)

        if days90 >= 30 and days_prev >= 30

        else 1.0
    )


    rows.append({

        "title_id": row.title_id,

        "hist_global_sessions_per_day_90d":
            global90[0] / max(days90, 1),

        "hist_genre_sessions_per_day_90d":
            recent_rate,

        "hist_genre_sessions_per_day_365d":
            genre365[0] / max(days365, 1),

        "hist_genre_session_support_365d":
            genre365[0],

        "hist_genre_completion_rate_365d":
            (
                genre365[1]
                + SMOOTHING * overall_completion
            )
            / (genre365[0] + SMOOTHING),

        "hist_genre_watch_fraction_365d":
            (
                genre365[2]
                + SMOOTHING * overall_fraction
            )
            / (genre365[0] + SMOOTHING),

        "hist_genre_trend_90d":
            trend,

        "hist_genre_share_90d":
            recent[0] / max(global90[0], 1),

        "hist_type_sessions_per_day_365d":
            content365[0] / max(days365, 1),

        "hist_type_completion_rate_365d":
            (
                content365[1]
                + SMOOTHING * overall_completion
            )
            / (content365[0] + SMOOTHING)

    })



# Merge features into the existing modelling table.

historical_features = pd.DataFrame(rows)

titles = titles.merge(
    historical_features,
    on="title_id",
    how="left",
    validate="one_to_one"
)

historical_feature_columns = [
    c
    for c in historical_features.columns
    if c != "title_id"
]


# Check that the new predictors contain valid numeric values.

assert (
    titles[historical_feature_columns]
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    titles[historical_feature_columns].to_numpy()
).all()


print("Historical sessions:", len(historical_sessions))
print("New features:", len(historical_feature_columns))

display(
    titles[historical_feature_columns]
    .describe()
    .T
    .round(3)
)

KeyError: "None of [Index(['hist_global_sessions_per_day_90d', 'hist_genre_sessions_per_day_90d',\n       'hist_genre_sessions_per_day_365d', 'hist_genre_session_support_365d',\n       'hist_genre_completion_rate_365d', 'hist_genre_watch_fraction_365d',\n       'hist_genre_trend_90d', 'hist_genre_share_90d',\n       'hist_type_sessions_per_day_365d', 'hist_type_completion_rate_365d'],\n      dtype='object')] are in the [columns]"

In [ ]:
# Attach actors to historical viewing sessions.

actor_session_history = (
    historical_sessions[
        [
            "title_id",
            "session_end",
            "completed_session",
            "watch_fraction"
        ]
    ]
    .merge(
        titles[["title_id", "actor_list"]],
        on="title_id",
        how="left",
        validate="many_to_one"
    )
    .explode("actor_list")
    .dropna(subset=["actor_list"])
    .rename(
        columns={"actor_list": "history_key"}
    )
)

actor_history = build_lookup(
    actor_session_history,
    "history_key"
)


actor_rows = []

for row in titles[
    ["title_id", "acquisition_date", "actor_list"]
].itertuples(index=False):

    actor_stats = [
        prior_stats(
            actor_history,
            actor,
            row.acquisition_date,
            365
        )
        for actor in row.actor_list
    ]

    average_actor = (
        np.mean(actor_stats, axis=0)
        if actor_stats
        else np.zeros(3)
    )

    global_stats = prior_stats(
        all_history,
        "all",
        row.acquisition_date,
        365
    )

    global_completion = (
        (global_stats[1] + 1)
        / (global_stats[0] + 2)
    )

    smoothing = 20.0

    actor_rows.append({
        "title_id": row.title_id,

        "hist_actor_sessions_per_day_365d":
            average_actor[0]
            / max(available_days(row.acquisition_date, 365), 1),

        "hist_actor_completion_rate_365d":
            (
                average_actor[1]
                + smoothing * global_completion
            )
            / (average_actor[0] + smoothing),

        "hist_actor_session_support_365d":
            average_actor[0]
    })


actor_history_features = pd.DataFrame(actor_rows)

titles = titles.merge(
    actor_history_features,
    on="title_id",
    how="left",
    validate="one_to_one"
)

actor_history_feature_columns = [
    c for c in actor_history_features.columns
    if c != "title_id"
]

,count,mean,std,min,25%,50%,75%,max
hist_actor_sessions_per_day_365d,1140.0,0.165699,0.184706,0.0,0.0,0.0,0.350685,0.726027
hist_actor_completion_rate_365d,1140.0,0.639972,0.144147,0.5,0.5,0.5,0.785603,0.853659
hist_actor_session_support_365d,1140.0,52.518713,64.590748,0.0,0.0,0.0,122.000000,265.000000


### Integrate search data to predict demand

In [ ]:
search = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vSUrURjf8tkeH8bpCNauhmu2PSOyxnjp3OHhHp9Zsd-YYhRASUknPJgaT9es1ZqQCyxxEBt0vdeo5Co/pub?gid=195541149&single=true&output=csv").drop_duplicates()

search["search_at"] = pd.to_datetime(
    search["search_timestamp"],
    unit="s",
    utc=True
).dt.tz_localize(None)


def normalise_search_text(value):
    text = str(value).casefold()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


# Strip common search suffixes, but only after checking whether the original query matches exactly.

suffix_pattern = re.compile(
    r"\s+(?:trailer|cast|full\s+movie|full\s+moive|"
    r"episode\s*\d+|season\s*\d+)$"
)


def strip_search_suffixes(query):
    while True:
        updated = suffix_pattern.sub("", query).strip()

        if updated == query:
            return query

        query = updated


titles["title_search_key"] = (
    titles["title_name"].map(normalise_search_text)
)

title_lookup = dict(
    zip(
        titles["title_search_key"],
        titles["title_id"]
    )
)

candidate_names = list(title_lookup)


def match_search_query(query):

    if not query:
        return None, "unmatched", 0.0

    # First: exact match without modifying the query.

    if query in title_lookup:
        return title_lookup[query], "exact", 100.0

    # Second: remove common search qualifiers.

    cleaned = strip_search_suffixes(query)

    if cleaned in title_lookup:
        return title_lookup[cleaned], "suffix_exact", 100.0

    # Third: conservative fuzzy matching.

    if len(cleaned) < 7 or len(cleaned.split()) < 2:
        return None, "unmatched", 0.0

    candidates = process.extract(
        cleaned,
        candidate_names,
        scorer=fuzz.ratio,
        limit=2
    )

    if not candidates:
        return None, "unmatched", 0.0

    best_name, best_score, _ = candidates[0]

    second_score = (
        candidates[1][1]
        if len(candidates) > 1
        else 0
    )

    # Require a strong, unambiguous match.

    if (
        best_score >= 92
        and best_score - second_score >= 8
        and abs(len(cleaned) - len(best_name)) <= 3
    ):
        return (
            title_lookup[best_name],
            "fuzzy",
            float(best_score)
        )

    return None, "unmatched", float(best_score)


search["query_key"] = (
    search["search_term"].map(normalise_search_text)
)

match_cache = {
    query: match_search_query(query)
    for query in search["query_key"].unique()
}

search[
    ["matched_title_id", "match_method", "match_score"]
] = pd.DataFrame(
    search["query_key"].map(match_cache).tolist(),
    index=search.index
)



,count
match_method,
exact,4123
suffix_exact,2104
unmatched,1000
fuzzy,21


,search_term,matched_title_id,match_score
129,Nortehrn Current,stad-ser-1711,93.750000
168,Distant Seson Origins,stad-ser-1004,97.674419
498,Nortehrn Crossing season 2,stad-ser-1093,94.117647
533,Northern Seson Returns,stad-mov-1907,97.777778
833,Teh Reckoning,stad-ser-1363,92.307692
1139,Teh Mountain Origins,stad-ser-1328,95.000000
1578,Second Seson,stad-doc-2041,96.000000
1889,Bitter Seson II,stad-mov-1652,96.774194
2724,Nortehrn Reckoning Chronicles,stad-mov-1029,96.551724
2769,Golden Seson Uncovered,stad-ser-1461,97.777778


In [ ]:
# Attach acquisition dates to successfully matched searches.

matched_searches = (
    search.loc[
        search["matched_title_id"].notna()
    ]
    .merge(
        titles[["title_id", "acquisition_date"]],
        left_on="matched_title_id",
        right_on="title_id",
        how="inner",
        validate="many_to_one"
    )
)

matched_searches["days_before_acquisition"] = (
    (
        matched_searches["acquisition_date"]
        - matched_searches["search_at"]
    ).dt.total_seconds() / 86400
)


# Exclude searches on or after acquisition. Retain only the prior 365 days.

prior_searches = matched_searches.loc[
    (matched_searches["days_before_acquisition"] > 0)
    & (matched_searches["days_before_acquisition"] <= 365)
].copy()

prior_searches["within_90d"] = (
    prior_searches["days_before_acquisition"] <= 90
).astype(int)


# Aggregate title-specific search counts.

search_counts = (
    prior_searches
    .groupby("matched_title_id")
    .agg(
        search_count_365d=("session_id", "size"),
        search_count_90d=("within_90d", "sum")
    )
)

titles = titles.join(
    search_counts,
    on="title_id"
)

for column in [
    "search_count_90d",
    "search_count_365d"
]:
    titles[column] = (
        titles[column]
        .fillna(0)
        .astype(float)
    )


# Calculate total platform searches within each time window. This provides a denominator for relative demand.

all_search_times = np.sort(
    search["search_at"].to_numpy(
        dtype="datetime64[ns]"
    )
)

acquisition_times = (
    titles["acquisition_date"]
    .to_numpy(dtype="datetime64[ns]")
)

search_history_start = (
    search["search_at"].min().floor("D")
)


for days in [90, 365]:

    total_searches = (
        np.searchsorted(
            all_search_times,
            acquisition_times,
            side="left"
        )
        -
        np.searchsorted(
            all_search_times,
            acquisition_times - np.timedelta64(days, "D"),
            side="left"
        )
    )

    coverage_days = np.array([
        max(
            0.0,
            (
                acquisition_date
                - max(
                    acquisition_date - pd.Timedelta(days=days),
                    search_history_start
                )
            ).total_seconds() / 86400
        )
        for acquisition_date in titles["acquisition_date"]
    ])

    titles[f"search_coverage_days_{days}d"] = coverage_days

    title_counts = titles[
        f"search_count_{days}d"
    ].to_numpy()


    # Searches per 30 days, where at least 30 days
    # of historical data are available.

    titles[f"search_rate_per30d_{days}d"] = np.where(
        coverage_days >= 30,
        title_counts / np.maximum(coverage_days, 1) * 30,
        np.nan
    )


    # Candidate title's share of platform searches.

    titles[f"search_share_{days}d"] = np.where(
        total_searches >= 30,
        title_counts / np.maximum(total_searches, 1),
        np.nan
    )


search_feature_columns = [
    "search_count_90d",
    "search_count_365d",
    "search_rate_per30d_90d",
    "search_rate_per30d_365d",
    "search_share_90d",
    "search_share_365d",
    "search_coverage_days_90d",
    "search_coverage_days_365d"
]

Pre-acquisition searches within 365 days: 706
Titles with prior search demand: 255


,count,mean,std,min,25%,50%,75%,max
search_count_90d,1140.0,0.200877,0.566088,0.0,0.0,0.000000,0.000000,4.000000
search_count_365d,1140.0,0.619298,1.485303,0.0,0.0,0.000000,0.000000,9.000000
search_rate_per30d_90d,353.0,0.223059,0.301081,0.0,0.0,0.000000,0.333333,1.333333
search_rate_per30d_365d,353.0,0.212624,0.222823,0.0,0.0,0.164384,0.328767,1.304348
search_share_90d,370.0,0.000759,0.001331,0.0,0.0,0.000000,0.001130,0.017857
search_share_365d,370.0,0.000722,0.001153,0.0,0.0,0.000550,0.001095,0.017857
search_coverage_days_90d,1140.0,27.490351,40.909211,0.0,0.0,0.000000,90.000000,90.000000
search_coverage_days_365d,1140.0,90.125439,145.955196,0.0,0.0,0.000000,166.250000,365.000000
